In [ ]:
# =====================================================================
# SCRIPT A — DATA PREP  →  prepared_sequences.csv (sequence, subtype, cluster)
# Run once on the downloaded LANL multi-subtype pol FASTA.
# =====================================================================
!pip install Biopython datasketch
import warnings; warnings.filterwarnings("ignore")
import random
import numpy as np, pandas as pd
from Bio import SeqIO
from datasketch import MinHash, MinHashLSH

SEED = 56
random.seed(SEED); np.random.seed(SEED)

# ---- KNOBS you may change -------------------------------------------------
FASTA_PATH    = "hiv-db.fasta"   # <-- your downloaded all-subtype pol FASTA
MIN_PER_CLASS = 100                  # drop any subtype with fewer sequences than this
JACCARD_THR   = 0.90                 # near-duplicate threshold for clustering
OUT_CSV       = "prepared_sequences.csv"
# ---------------------------------------------------------------------------

# ===== CELL 1: parse FASTA into a dataframe ===============================
# LANL headers look like "B.US.1983.HXB2.K03455" or "01_AE.TH.1990....".
# The subtype label is the first field before the first ".". This handles
# pure subtypes (A1, B, C, D...) AND CRFs (01_AE, 02_AG...) automatically.
VALID = set("ACGT")
def clean_seq(s):
    s = s.upper().replace("-", "").replace(".", "").replace(" ", "")
    return "".join(c if c in VALID else "N" for c in s)   # unknown bases -> N

rows = []
for rec in SeqIO.parse(FASTA_PATH, "fasta"):
    subtype = rec.description.split(".")[0].strip()        # first header field
    rows.append({"sequence": clean_seq(str(rec.seq)), "subtype": subtype})
df = pd.DataFrame(rows)
print(f"Parsed {len(df)} sequences, {df['subtype'].nunique()} raw subtype labels")

# ===== CELL 2: drop rare classes + exact duplicates ======================
# A classifier can't learn a subtype from 12 sequences, and rare classes
# break the stratified split later. Keep only well-represented subtypes.
counts = df["subtype"].value_counts()
keep   = counts[counts >= MIN_PER_CLASS].index
df     = df[df["subtype"].isin(keep)].copy()
df     = df.drop_duplicates("sequence").reset_index(drop=True)
print(f"After filtering (>= {MIN_PER_CLASS}/class) + dedup: {len(df)} sequences")
print(df["subtype"].value_counts())

# ===== CELL 3: MinHash signatures (k=7, codon stride, 64 permutations) ====
def seq_minhash(seq, k=7, num_perm=64):
    mh = MinHash(num_perm=num_perm, seed=SEED)
    for i in range(0, len(seq) - k + 1, 3):     # stride 3 = codon step
        mh.update(seq[i:i+k].encode("utf8"))
    return mh

mhs = []
for n, seq in enumerate(df["sequence"]):
    mhs.append(seq_minhash(seq))
    if n % 5000 == 0: print(f"  minhash {n}/{len(df)}")

# ===== CELL 4: cluster near-duplicates with LSH + union-find ==============
# This groups every set of >=90%-similar sequences under one cluster id.
# Union-find gives correct *transitive* grouping (A~B, B~C => A,B,C together),
# fixing a subtle bug in the original single-pass version.
n = len(df)
parent = list(range(n))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[ra] = rb

lsh = MinHashLSH(threshold=JACCARD_THR, num_perm=64)
for i, mh in enumerate(mhs): lsh.insert(i, mh)
for i, mh in enumerate(mhs):
    for j in lsh.query(mh):
        if j != i: union(i, j)

roots = [find(i) for i in range(n)]
# relabel roots to clean 0..C-1 ids
remap = {r: c for c, r in enumerate(sorted(set(roots)))}
df["cluster"] = [remap[r] for r in roots]
print(f"Clusters (non-redundant groups): {df['cluster'].nunique()}")

# ===== CELL 5: save =======================================================
df.to_csv(OUT_CSV, index=False)
print(f"Saved -> {OUT_CSV}  (rows={len(df)}, classes={df['subtype'].nunique()}, "
      f"clusters={df['cluster'].nunique()})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 3.4 MB/s eta 0:00:00
Parsed 38649 sequences, 361 raw subtype labels
After filtering (>= 100/class) + dedup: 26292 sequences
subtype
B        13142
01_AE     4105
C         4012
A1         797
07_BC      748
-          479
A1D        435
D          352
01B        291
08_BC      289
02_AG      272
BC         249
A6         238
A1C        197
BF1        159
A1CD       153
0107       141
G          117
01BC       116
Name: count, dtype: int64
  minhash 0/26292
  minhash 5000/26292
  minhash 10000/26292
  minhash 15000/26292
  minhash 20000/26292
  minhash 25000/26292
Clusters (non-redundant groups): 12742
Saved -> prepared_sequences.csv  (rows=26292, classes=19, clusters=12742)
